# 06 — Remaining Useful Life (RUL) Prediction
## Wind Turbine Gearbox Predictive Maintenance

Bu notebook, **Kalan Kullanım Ömrü (RUL)** tahminini ele alır. Anomali başlamadan önce kaç saat/gün kaldığını tahmin etmek, bakım planlaması için kritik öneme sahiptir.

**İçerik:**
1. RUL kavramı ve metodoloji
2. Degradation curve çıkarma
3. Weibull distribution fit
4. LSTM/GRU ile RUL regression
5. Early warning sistemi (24/48/72 saat öncesi)
6. Predictive maintenance timeline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import glob
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.optimize import curve_fit
from scipy.special import gamma
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 6)
sns.set_style('whitegrid')
print('Libraries loaded!')

## 1. RUL Kavramı

**Remaining Useful Life (RUL)**, bir makinenin belirli bir anda kaç saat/gün daha hatasız çalışabileceğinin tahminidir.

$$RUL_t = T_{failure} - t$$

Burada:
- $T_{failure}$: Anomali/arıza zamanı
- $t$: Şu anki zaman
- $RUL_t$: $t$ anında kalan ömür

**Yaklaşım:**
1. Anomali dönemlerini bul (hedef: anomali başlangıç noktaları)
2. Her anomali öncesinde RUL değerlerini hesapla (geri sayım)
3. Bu geri sayımı tahmin etmeye çalış

In [ ]:
# Veri yükle
PROCESSED_PATH = '../data/processed/features_engineered.csv'
DATA_PATH = '/kaggle/input/wind-turbine-gearbox-anomaly-detection-5year-scada/'

if os.path.exists(PROCESSED_PATH):
    df = pd.read_csv(PROCESSED_PATH, index_col=0, parse_dates=True)
else:
    csv_files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
    dfs = [pd.read_csv(f) for f in sorted(csv_files)]
    df = pd.concat(dfs, ignore_index=True)
    time_col = [c for c in df.columns if 'time' in c.lower() or 'date' in c.lower()]
    if time_col:
        df[time_col[0]] = pd.to_datetime(df[time_col[0]])
        df = df.sort_values(time_col[0]).set_index(time_col[0])

anomaly_col = [c for c in df.columns if 'anomal' in c.lower() or 'label' in c.lower() or 'fault' in c.lower()]
ANOMALY_COL = anomaly_col[0] if anomaly_col else df.columns[-1]

FEATURE_COLS = df.select_dtypes(include=[np.number]).columns.tolist()
if ANOMALY_COL in FEATURE_COLS:
    FEATURE_COLS.remove(ANOMALY_COL)

print(f'Dataset: {df.shape}, Anomaly rate: {df[ANOMALY_COL].mean()*100:.2f}%')

## 2. RUL Hesaplama

Her anomali döneminden önce geri sayım (countdown) oluşturuyoruz.

In [ ]:
def compute_rul(anomaly_series, max_rul=168):
    """Anomali serisinden RUL geri sayımı oluşturur.
    
    max_rul: Maksimum RUL değeri (örn: 168 saat = 1 hafta)
    Normal dönemlerde max_rul, anomali başladığında 0'a iner.
    """
    y = anomaly_series.values.astype(int)
    rul = np.full(len(y), max_rul, dtype=np.float32)
    
    # Anomali bloklarını bul
    i = len(y) - 1
    while i >= 0:
        if y[i] == 1:
            # Anomali başlangıcını bul
            end = i
            while i >= 0 and y[i] == 1:
                i -= 1
            start = i + 1
            
            # Anomali öncesinde geri sayım oluştur
            lookback = min(max_rul, start)
            for j in range(start - lookback, start):
                rul[j] = min(max_rul, start - j)
            
            # Anomali döneminde RUL = 0
            rul[start:end+1] = 0
        else:
            i -= 1
    
    return rul

MAX_RUL = 168  # 1 hafta
rul_values = compute_rul(df[ANOMALY_COL], max_rul=MAX_RUL)
df['RUL'] = rul_values

print(f'RUL statistics:')
print(f'  Min:  {df["RUL"].min():.0f}h')
print(f'  Max:  {df["RUL"].max():.0f}h')
print(f'  Mean: {df["RUL"].mean():.1f}h')
print(f'  RUL=0 count: {(df["RUL"]==0).sum():,} ({(df["RUL"]==0).mean()*100:.2f}%)')

# Görselleştir
sample_size = min(5000, len(df))
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(df.index[:sample_size], df[ANOMALY_COL].values[:sample_size],
             color='red', alpha=0.7, label='Anomaly Flag')
axes[0].set_title('Anomaly Timeline', fontsize=13)
axes[0].set_yticks([0, 1])
axes[0].legend()

axes[1].plot(df.index[:sample_size], df['RUL'].values[:sample_size],
             color='steelblue', linewidth=1, label='RUL (hours)')
axes[1].set_title(f'Remaining Useful Life (max={MAX_RUL}h)', fontsize=13)
axes[1].set_ylabel('RUL (hours)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/rul_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Degradation Curve Analizi

Anomali başlamadan önce sensörlerin nasıl değiştiğini (degradasyon eğrisi) inceliyoruz.

In [ ]:
# Anomali olaylarını tespit et
anomaly_events = []
in_anomaly = False
start_idx = None

for i, (idx, row) in enumerate(df.iterrows()):
    if row[ANOMALY_COL] == 1 and not in_anomaly:
        in_anomaly = True
        start_idx = i
    elif row[ANOMALY_COL] == 0 and in_anomaly:
        in_anomaly = False
        anomaly_events.append(start_idx)

print(f'Anomaly events found: {len(anomaly_events)}')

# Her anomali öncesindeki 168 saatlik degradasyon
LOOKBACK = 168
original_features = [c for c in FEATURE_COLS
                     if '_roll_' not in c and '_lag_' not in c
                     and '_sin_' not in c and '_cos_' not in c]

if len(anomaly_events) > 0 and original_features:
    # İlk sensör için degradasyon eğrisi
    sensor = original_features[0]
    degradation_curves = []
    
    for start in anomaly_events[:min(5, len(anomaly_events))]:
        if start >= LOOKBACK:
            curve = df[sensor].values[start-LOOKBACK:start]
            # Normalize
            curve_norm = (curve - curve.min()) / (curve.max() - curve.min() + 1e-8)
            degradation_curves.append(curve_norm)
    
    if degradation_curves:
        fig, ax = plt.subplots(figsize=(12, 5))
        t = np.arange(-LOOKBACK, 0)
        
        for i, curve in enumerate(degradation_curves):
            ax.plot(t, curve, alpha=0.5, linewidth=1, label=f'Event {i+1}')
        
        if degradation_curves:
            mean_curve = np.mean(degradation_curves, axis=0)
            ax.plot(t, mean_curve, color='red', linewidth=2.5, label='Mean degradation')
        
        ax.axvline(x=0, color='darkred', linestyle='--', label='Anomaly Start')
        ax.axvline(x=-24, color='orange', linestyle=':', label='-24h')
        ax.axvline(x=-48, color='yellow', linestyle=':', label='-48h')
        ax.set_title(f'{sensor} — Degradation Curves Before Anomaly Events', fontsize=13)
        ax.set_xlabel('Hours Before Anomaly')
        ax.set_ylabel('Normalized Value')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('results/degradation_curves.png', dpi=150, bbox_inches='tight')
        plt.show()

## 4. Weibull Distribution Fit

Weibull dağılımı, makine güvenilirlik analizinde standart bir araçtır. Arıza sürelerini modellemek için kullanılır.

$$f(t) = \frac{k}{\lambda}\left(\frac{t}{\lambda}\right)^{k-1}e^{-(t/\lambda)^k}$$

- **k < 1**: Erken dönem arızaları (infant mortality)
- **k = 1**: Rastgele arızalar (exponential)
- **k > 1**: Yaşlanma ile artan arıza oranı (wear-out)

In [ ]:
# Anomali olayları arası süreleri hesapla (time-to-failure)
if len(anomaly_events) > 1:
    inter_event_times = np.diff(anomaly_events).astype(float)
    inter_event_times = inter_event_times[inter_event_times > 0]
    
    if len(inter_event_times) > 3:
        # Weibull fit (scipy kullanarak)
        shape, loc, scale = stats.weibull_min.fit(inter_event_times, floc=0)
        print(f'Weibull Parameters:')
        print(f'  Shape (k): {shape:.3f}')
        print(f'  Scale (λ): {scale:.1f} hours')
        
        if shape < 1:
            print('  → k < 1: Decreasing failure rate (early life / bathtub left)')
        elif shape == 1:
            print('  → k = 1: Constant failure rate (exponential)')
        else:
            print('  → k > 1: Increasing failure rate (wear-out)')
        
        # Görselleştir
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Histogram + fit
        x = np.linspace(inter_event_times.min(), inter_event_times.max(), 200)
        pdf_fit = stats.weibull_min.pdf(x, shape, loc=0, scale=scale)
        
        axes[0].hist(inter_event_times, bins=20, density=True,
                    alpha=0.6, color='steelblue', label='Observed')
        axes[0].plot(x, pdf_fit, color='red', linewidth=2.5, label='Weibull fit')
        axes[0].set_title('Time Between Anomaly Events — Weibull Fit', fontsize=12)
        axes[0].set_xlabel('Hours')
        axes[0].set_ylabel('Density')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Survival function (Reliability)
        survival = stats.weibull_min.sf(x, shape, loc=0, scale=scale)
        axes[1].plot(x, survival * 100, color='green', linewidth=2.5)
        axes[1].axhline(y=50, color='orange', linestyle='--', label='50% reliability')
        axes[1].set_title('Weibull Reliability Function', fontsize=12)
        axes[1].set_xlabel('Hours')
        axes[1].set_ylabel('Reliability (%)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        mean_life = scale * gamma(1 + 1/shape)
        print(f'\nMTTF (Mean Time to Failure): {mean_life:.1f} hours = {mean_life/24:.1f} days')
        
        plt.tight_layout()
        plt.savefig('results/weibull_fit.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print(f'Not enough events for Weibull fit. Found: {len(inter_event_times)}')
else:
    print('Not enough anomaly events for inter-event analysis.')

## 5. LSTM/GRU ile RUL Regression

Sliding window ile geçmiş sensör değerlerinden RUL tahmin eden bir regression modeli.

In [ ]:
# RUL dataset oluştur — sadece RUL < max_rul olan örnekler
WINDOW_SIZE = 48

# Feature scaling
feature_scaler = StandardScaler()
rul_scaler = MinMaxScaler()

X_features = df[FEATURE_COLS].replace([np.inf, -np.inf], np.nan).fillna(0).values
y_rul = df['RUL'].values

# Temporal split
split_idx = int(len(X_features) * 0.8)
X_features_scaled = feature_scaler.fit_transform(X_features)
y_rul_scaled = rul_scaler.fit_transform(y_rul.reshape(-1, 1)).flatten()

def create_rul_windows(X, y, window_size=48, max_rul=168):
    """RUL < max_rul örnekler için sliding windows."""
    X_windows, y_windows, indices = [], [], []
    for i in range(window_size, len(X)):
        # Sadece degradasyon bölgesindeki örnekler
        if y[i] < max_rul:
            X_windows.append(X[i-window_size:i])
            y_windows.append(y[i])
            indices.append(i)
    return np.array(X_windows, dtype=np.float32), np.array(y_windows, dtype=np.float32)

X_win, y_win = create_rul_windows(X_features_scaled, y_rul_scaled, WINDOW_SIZE, 1.0)
print(f'RUL dataset: {X_win.shape}, labels: {y_win.shape}')

# Split
tr_size = int(len(X_win) * 0.7)
val_size = int(len(X_win) * 0.15)

X_tr_r = X_win[:tr_size]
y_tr_r = y_win[:tr_size]
X_val_r = X_win[tr_size:tr_size+val_size]
y_val_r = y_win[tr_size:tr_size+val_size]
X_te_r = X_win[tr_size+val_size:]
y_te_r = y_win[tr_size+val_size:]
print(f'Train: {X_tr_r.shape} | Val: {X_val_r.shape} | Test: {X_te_r.shape}')

In [ ]:
def build_lstm_rul(window_size, n_features, units=128, dropout=0.3):
    """LSTM RUL regression modeli."""
    inputs = keras.Input(shape=(window_size, n_features))
    x = layers.LSTM(units, return_sequences=True, dropout=dropout)(inputs)
    x = layers.LSTM(units // 2, return_sequences=False, dropout=dropout)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)  # [0,1] → denormalize
    return keras.Model(inputs, outputs, name='LSTM_RUL')

def build_gru_rul(window_size, n_features, units=128, dropout=0.3):
    """GRU RUL regression modeli."""
    inputs = keras.Input(shape=(window_size, n_features))
    x = layers.GRU(units, return_sequences=True, dropout=dropout)(inputs)
    x = layers.GRU(units // 2, return_sequences=False, dropout=dropout)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inputs, outputs, name='GRU_RUL')

n_features_rul = X_win.shape[2]
rul_callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

# LSTM eğit
lstm_rul = build_lstm_rul(WINDOW_SIZE, n_features_rul)
lstm_rul.compile(optimizer='adam', loss='huber', metrics=['mae'])
lstm_rul_history = lstm_rul.fit(
    X_tr_r, y_tr_r, validation_data=(X_val_r, y_val_r),
    epochs=50, batch_size=256, callbacks=rul_callbacks, verbose=1
)

# GRU eğit
gru_rul = build_gru_rul(WINDOW_SIZE, n_features_rul)
gru_rul.compile(optimizer='adam', loss='huber', metrics=['mae'])
gru_rul_history = gru_rul.fit(
    X_tr_r, y_tr_r, validation_data=(X_val_r, y_val_r),
    epochs=50, batch_size=256, callbacks=rul_callbacks, verbose=1
)

In [ ]:
# Değerlendirme
def evaluate_rul_model(model, X_test, y_test_scaled, rul_scaler, model_name):
    y_pred_scaled = model.predict(X_test, verbose=0).flatten()
    
    # Denormalize
    y_pred = rul_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true = rul_scaler.inverse_transform(y_test_scaled.reshape(-1, 1)).flatten()
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f'{model_name}:')
    print(f'  MAE:  {mae:.2f} hours ({mae/24:.1f} days)')
    print(f'  RMSE: {rmse:.2f} hours ({rmse/24:.1f} days)')
    print(f'  R²:   {r2:.4f}')
    
    return y_true, y_pred, {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

y_true_lstm, y_pred_lstm, lstm_metrics = evaluate_rul_model(lstm_rul, X_te_r, y_te_r, rul_scaler, 'LSTM')
y_true_gru, y_pred_gru, gru_metrics = evaluate_rul_model(gru_rul, X_te_r, y_te_r, rul_scaler, 'GRU')

# Gerçek vs tahmin grafiği
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (y_true, y_pred, name) in zip(axes, [
    (y_true_lstm, y_pred_lstm, 'LSTM'),
    (y_true_gru, y_pred_gru, 'GRU')
]):
    ax.scatter(y_true[:500], y_pred[:500], alpha=0.3, s=5, color='steelblue')
    perfect = [0, MAX_RUL]
    ax.plot(perfect, perfect, 'r--', linewidth=2, label='Perfect prediction')
    ax.set_title(f'{name} — Actual vs Predicted RUL', fontsize=12)
    ax.set_xlabel('Actual RUL (hours)')
    ax.set_ylabel('Predicted RUL (hours)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/rul_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Early Warning Sistemi (24/48/72 saat)

Model, anomali başlamadan 24, 48 veya 72 saat önce uyarı verebiliyor mu?

In [ ]:
def compute_early_warning_accuracy(y_true, y_pred, warning_horizons=[24, 48, 72]):
    """N saat öncesinde doğru alarm verme oranını hesapla."""
    results = {}
    for h in warning_horizons:
        # Gerçekte h saatlik warning zone (RUL ≤ h)
        true_warning = y_true <= h
        # Model de h saatlik zone'da mı?
        pred_warning = y_pred <= h
        
        true_positives = (true_warning & pred_warning).sum()
        total_warnings = true_warning.sum()
        
        if total_warnings > 0:
            accuracy = true_positives / total_warnings * 100
        else:
            accuracy = 0.0
        
        results[f'{h}h'] = accuracy
    return results

print('=== EARLY WARNING ACCURACY ===')
lstm_ew = compute_early_warning_accuracy(y_true_lstm, y_pred_lstm)
gru_ew = compute_early_warning_accuracy(y_true_gru, y_pred_gru)

ew_df = pd.DataFrame({'LSTM': lstm_ew, 'GRU': gru_ew})
print(ew_df.round(2))

# Görselleştir
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(ew_df))
width = 0.35

ax.bar(x - width/2, ew_df['LSTM'], width, color='#3498db', alpha=0.8, label='LSTM')
ax.bar(x + width/2, ew_df['GRU'], width, color='#e74c3c', alpha=0.8, label='GRU')
ax.set_title('Early Warning Accuracy by Horizon', fontsize=13)
ax.set_xlabel('Warning Horizon')
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(x)
ax.set_xticklabels(ew_df.index)
ax.set_ylim(0, 100)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('results/early_warning_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Predictive Maintenance Timeline

In [ ]:
# Maintenance decision zones
CRITICAL_ZONE = 24   # < 24h → Critical
WARNING_ZONE = 72    # 24-72h → Warning
MONITOR_ZONE = 168   # 72-168h → Monitor

# Son 500 test örneği için timeline
n_show = min(500, len(y_true_lstm))
x_timeline = np.arange(n_show)

fig, ax = plt.subplots(figsize=(16, 6))

# Bölge renkleri
ax.fill_between(x_timeline,
                [MONITOR_ZONE] * n_show, [MAX_RUL] * n_show,
                alpha=0.15, color='green', label=f'Normal (RUL > {MONITOR_ZONE}h)')
ax.fill_between(x_timeline,
                [WARNING_ZONE] * n_show, [MONITOR_ZONE] * n_show,
                alpha=0.15, color='yellow', label=f'Monitor ({WARNING_ZONE}h–{MONITOR_ZONE}h)')
ax.fill_between(x_timeline,
                [CRITICAL_ZONE] * n_show, [WARNING_ZONE] * n_show,
                alpha=0.15, color='orange', label=f'Warning ({CRITICAL_ZONE}h–{WARNING_ZONE}h)')
ax.fill_between(x_timeline,
                [0] * n_show, [CRITICAL_ZONE] * n_show,
                alpha=0.15, color='red', label=f'Critical (< {CRITICAL_ZONE}h)')

# Gerçek ve tahmin
ax.plot(x_timeline, y_true_lstm[:n_show], color='black', linewidth=1.5,
        alpha=0.8, label='Actual RUL')
ax.plot(x_timeline, y_pred_lstm[:n_show], color='blue', linewidth=1.5,
        alpha=0.7, linestyle='--', label='Predicted RUL (LSTM)')

ax.set_title('Predictive Maintenance Timeline — RUL Tracking', fontsize=14)
ax.set_xlabel('Time Steps')
ax.set_ylabel('Remaining Useful Life (hours)')
ax.set_ylim(0, MAX_RUL + 10)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/maintenance_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Özet
rul_comparison = pd.DataFrame([lstm_metrics, gru_metrics]).set_index('Model')
print('=== RUL PREDICTION SUMMARY ===')
print(rul_comparison.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
metrics_to_plot = ['MAE', 'RMSE']
rul_comparison[metrics_to_plot].plot(kind='bar', ax=ax, color=['#3498db', '#e74c3c'])
ax.set_title('RUL Model Comparison — MAE and RMSE (hours)', fontsize=13)
ax.set_ylabel('Error (hours)')
ax.tick_params(axis='x', rotation=0)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('results/rul_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ RUL Prediction Complete!')

## Özet

Bu notebook'ta:
- RUL geri sayım değerleri hesaplandı
- Degradation curve analizi yapıldı
- Weibull distribution ile MTTF tahmin edildi
- LSTM ve GRU ile RUL regression eğitildi
- 24/48/72 saatlik early warning doğruluğu ölçüldü
- Predictive maintenance timeline oluşturuldu

**Bu proje için en değerli tez katkısı:** RUL tahmini, anomali tespitinden bir adım öteye geçerek *ne zaman bakım gerektiğini* söyler.